# E032 — Quality-Aware Multimodal Fusion

Current fusion (E027) uses **fixed weights** (mfcc=0.02, lpcc=0.60, image=0.38) optimized on OOF.
This experiment tests **quality-aware fusion**: dynamically weight each modality per-sample based
on estimated quality metrics.

**Motivation:** The QME paper (2025) shows quality-guided score fusion improves multimodal biometric
recognition by downweighting low-quality inputs. Our image system is brittle to geometric degradations
(E028: rotation ±15° → 7.3% EER), and audio may have varying noise levels.

**Quality metrics:**
- Image: Laplacian variance (sharpness), brightness deviation from training mean
- Audio: SNR estimate, zero-crossing rate (noise indicator)

**Fusion strategies:**
1. `fixed` — E027 baseline (fixed weights)
2. `quality_softmax` — quality scores → softmax weights per sample
3. `quality_threshold` — drop modality if quality < threshold, renormalize
4. `quality_linear` — linear scaling: w_m = base_w * (quality_m / mean_quality_m)

**Hypothesis:** Quality-aware fusion reduces EER by downweighting degraded modalities per-sample,
especially for geometric image degradations and noisy audio.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from PIL import Image, ImageFilter
from scipy.special import logsumexp
from scipy.ndimage import laplace
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED           = 67
UBM_COMPONENTS = 32
MAP_R          = 16.0
SNR_DB         = 20.0
N_PCA          = 50
C_LOGREG       = 1.0

COLORS = {
    'target': '#E74C3C',
    'nontarget': '#2E86AB',
    'green': '#27AE60',
    'purple': '#8E44AD',
    'gray': '#95A5A6',
}
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target')

E027_REF = {'oof_eer': 0.26, 'oof_min_dcf': 0.0052, 'weights': (0.02, 0.60, 0.38)}

## 1. Quality metric functions

In [ ]:
def image_sharpness(img_arr):
    """Laplacian variance — higher = sharper."""
    if img_arr.ndim == 3:
        img_arr = img_arr.mean(axis=2)
    lap = laplace(img_arr.astype(np.float64))
    return float(np.var(lap))

def image_brightness(img_arr):
    """Mean brightness (0-255)."""
    if img_arr.ndim == 3:
        img_arr = img_arr.mean(axis=2)
    return float(np.mean(img_arr))

def audio_snr_estimate(y):
    """Simple SNR estimate: signal power / noise power (high-freq residual).
    Higher = cleaner audio.
    """
    # Estimate noise as high-frequency content (>4kHz)
    sr = 16000  # assumed
    n_fft = 512
    D = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=n_fft//4))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    noise_mask = freqs > 4000
    signal_power = np.mean(D**2)
    noise_power = np.mean(D[noise_mask, :]**2) + 1e-10
    return float(10 * np.log10(signal_power / noise_power))

def audio_zcr(y, sr=16000):
    """Zero-crossing rate — higher = more noise-like."""
    zcr = librosa.feature.zero_crossing_rate(y)
    return float(np.mean(zcr))

def normalize_quality(values, higher_better=True):
    """Normalize to [0, 1] range, 1 = best quality."""
    values = np.array(values)
    vmin, vmax = values.min(), values.max()
    if vmax - vmin < 1e-10:
        return np.ones_like(values)
    if higher_better:
        return (values - vmin) / (vmax - vmin)
    else:
        return 1.0 - (values - vmin) / (vmax - vmin)

print('Quality functions defined')

## 2. Model training and scoring (copy from E027/predict_fusion.py)

In [ ]:
import copy

def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _find_png(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

# MFCC extraction (E008)
def _extract_mfcc(y, sr):
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat   = np.vstack([mfcc, delta, delta2]).T
    feat  -= feat.mean(axis=0)
    return feat

def _aug_noise(y, rng):
    p = np.mean(y**2) + 1e-10
    return y + rng.normal(0, np.sqrt(p / 10**(SNR_DB/10)), len(y)).astype(y.dtype)

def _aug_speed(y, rng):
    return librosa.effects.time_stretch(y, rate=rng.uniform(0.9, 1.1))

# LPCC extraction (E025)
def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat   = np.array(cep_frames, dtype=np.float32)
    delta  = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat   = np.hstack([feat, delta, delta2])
    feat  -= feat.mean(axis=0)
    return feat

def _aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

# UBM-MAP
def _train_ubm(X):
    return GaussianMixture(n_components=UBM_COMPONENTS, covariance_type="diag",
                           max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp  = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp      = np.exp(log_resp)
    n_k       = resp.sum(axis=0)
    mu_hat    = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha     = n_k / (n_k + MAP_R)
    adapted   = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

def _train_mfcc(df, data_dir, augment, seed):
    rng = np.random.default_rng(seed)
    X_list, y_list = [], []
    for _, row in df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        wavs = [y_wav]
        if augment:
            wavs += [_aug_noise(y_wav, rng), _aug_speed(y_wav, rng)]
        for y_aug in wavs:
            f = _extract_mfcc(y_aug, sr)
            X_list.append(f); y_list.extend([row["label"]] * len(f))
    X = np.vstack(X_list); y = np.array(y_list)
    ubm     = _train_ubm(X[y == 0])
    adapted = _map_adapt(ubm, X[y == 1])
    return ubm, adapted

def _train_lpcc(df, data_dir, augment, seed):
    rng = np.random.default_rng(seed)
    X_list, y_list = [], []
    for _, row in df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        wavs = [y_wav]
        if augment:
            wavs += [_aug_pitch(y_wav, sr, rng)]
        for y_aug in wavs:
            f = _extract_lpcc(y_aug, sr)
            X_list.append(f); y_list.extend([row["label"]] * len(f))
    X = np.vstack(X_list); y = np.array(y_list)
    ubm     = _train_ubm(X[y == 0])
    adapted = _map_adapt(ubm, X[y == 1])
    return ubm, adapted

def _llr_mfcc(feat, adapted, ubm):
    ll_target   = adapted.score(feat)
    ll_ubm      = ubm.score(feat)
    return float(ll_target - ll_ubm)

def _llr_lpcc(feat, adapted, ubm):
    ll_target   = adapted.score(feat)
    ll_ubm      = ubm.score(feat)
    return float(ll_target - ll_ubm)

def _score_mfcc(wav_path, adapted, ubm):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat  = _extract_mfcc(y, sr)
    return _llr_mfcc(feat, adapted, ubm)

def _score_lpcc(wav_path, adapted, ubm):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat  = _extract_lpcc(y, sr)
    return _llr_lpcc(feat, adapted, ubm)

# Image (E007)
def _load_image(path):
    img = Image.open(path).convert("RGB")
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2).flatten()

def _aug_flip(x): return x.reshape(80, 80)[:, ::-1].flatten()
def _aug_bright(x, rng): return np.clip(x * rng.uniform(0.7, 1.3), 0, 255)
def _aug_inoise(x, rng): return np.clip(x + rng.normal(0, 15, x.shape), 0, 255)

def _train_image(df, data_dir, augment, seed):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row["stem"], data_dir))
        vecs = [orig]
        if augment:
            vecs += [_aug_flip(orig), _aug_bright(orig, rng), _aug_inoise(orig, rng)]
        for v in vecs:
            X.append(v); y.append(row["label"])
    X = np.stack(X); y = np.array(y)
    scaler = StandardScaler()
    pca    = PCA(n_components=N_PCA, random_state=SEED)
    clf    = LogisticRegression(C=C_LOGREG, max_iter=1000, random_state=SEED)
    X_pca  = pca.fit_transform(scaler.fit_transform(X))
    clf.fit(X_pca, y)
    return scaler, pca, clf

def _score_image(png_path, scaler, pca, clf):
    x = _load_image(png_path).reshape(1, -1)
    return float(clf.decision_function(pca.transform(scaler.transform(x)))[0])

print('Model functions defined')

## 3. Collect OOF scores + quality metrics

In [ ]:
print("Collecting OOF scores and quality metrics (3 folds)...")

oof_mfcc   = np.full(len(manifest), np.nan)
oof_lpcc   = np.full(len(manifest), np.nan)
oof_image  = np.full(len(manifest), np.nan)

# Quality metrics (computed once per sample, not per-fold)
q_sharpness = np.zeros(len(manifest))
q_brightness = np.zeros(len(manifest))
q_snr = np.zeros(len(manifest))
q_zcr = np.zeros(len(manifest))

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    seed_f   = SEED + fold_id
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]
    print(f"  fold {fold_id} ({len(val_idx)} val samples)...")

    ubm_m, ad_m = _train_mfcc(train_df, DATA, augment=True, seed=seed_f)
    ubm_l, ad_l = _train_lpcc(train_df, DATA, augment=True, seed=seed_f)
    scaler, pca, clf = _train_image(train_df, DATA, augment=True, seed=seed_f)

    for idx, row in val_df.iterrows():
        wav_path = _find_wav(row["stem"], DATA)
        png_path = _find_png(row["stem"], DATA)
        
        oof_mfcc[idx]  = _score_mfcc(wav_path, ad_m, ubm_m)
        oof_lpcc[idx]  = _score_lpcc(wav_path, ad_l, ubm_l)
        oof_image[idx] = _score_image(png_path, scaler, pca, clf)
        
        # Compute quality metrics (once per sample)
        y_wav, sr = librosa.load(wav_path, sr=None, mono=True)
        img_arr = np.array(Image.open(png_path).convert("RGB"))
        
        q_sharpness[idx] = image_sharpness(img_arr)
        q_brightness[idx] = image_brightness(img_arr)
        q_snr[idx] = audio_snr_estimate(y_wav)
        q_zcr[idx] = audio_zcr(y_wav, sr)

print(f"OOF collection complete.")
print(f"Sharpness: {q_sharpness.min():.1f} – {q_sharpness.max():.1f} (mean={q_sharpness.mean():.1f})")
print(f"Brightness: {q_brightness.min():.1f} – {q_brightness.max():.1f} (mean={q_brightness.mean():.1f})")
print(f"SNR: {q_snr.min():.1f} – {q_snr.max():.1f} dB (mean={q_snr.mean():.1f} dB)")
print(f"ZCR: {q_zcr.min():.4f} – {q_zcr.max():.4f} (mean={q_zcr.mean():.4f})")

## 4. Calibration + quality-aware fusion

In [ ]:
def _fit_calibrator(scores, labels):
    cal = LogisticRegression(C=1e6, max_iter=1000, class_weight="balanced")
    cal.fit(scores.reshape(-1, 1), labels)
    return cal

print("Fitting calibrators...")
cal_m = _fit_calibrator(oof_mfcc,  y_all)
cal_l = _fit_calibrator(oof_lpcc,  y_all)
cal_i = _fit_calibrator(oof_image, y_all)

cal_mo = cal_m.decision_function(oof_mfcc.reshape(-1, 1))
cal_lo = cal_l.decision_function(oof_lpcc.reshape(-1, 1))
cal_io = cal_i.decision_function(oof_image.reshape(-1, 1))

# Normalize quality metrics to [0, 1]
q_img_sharp = normalize_quality(q_sharpness, higher_better=True)
q_img_bright = 1.0 - np.abs(q_brightness - q_brightness.mean()) / (q_brightness.max() - q_brightness.min())
q_audio_snr = normalize_quality(q_snr, higher_better=True)
q_audio_zcr = normalize_quality(q_zcr, higher_better=False)

# Combined quality per modality
q_image  = 0.7 * q_img_sharp + 0.3 * q_img_bright
q_audio  = 0.7 * q_audio_snr + 0.3 * q_audio_zcr
# MFCC and LPCC share same audio, so same quality
q_mfcc   = q_audio.copy()

print(f"Image quality: {q_image.min():.3f} – {q_image.max():.3f} (mean={q_image.mean():.3f})")
print(f"Audio quality: {q_audio.min():.3f} – {q_audio.max():.3f} (mean={q_audio.mean():.3f})")

## 5. Compare fusion strategies

In [ ]:
from scipy.special import softmax

# E027 baseline weights
W_M_BASE, W_L_BASE, W_I_BASE = 0.02, 0.60, 0.38

def fuse_fixed(cal_m, cal_l, cal_i, w_m, w_l, w_i):
    return w_m * cal_m + w_l * cal_l + w_i * cal_i

def fuse_quality_softmax(cal_m, cal_l, cal_i, q_m, q_l, q_i, base_w_m, base_w_l, base_w_i):
    """Quality → softmax weights per sample."""
    weights = np.zeros((len(cal_m), 3))
    weights[:, 0] = base_w_m * q_m
    weights[:, 1] = base_w_l * q_l
    weights[:, 2] = base_w_i * q_i
    # Softmax across modalities per sample
    weights_norm = softmax(weights, axis=1)
    fused = (weights_norm[:, 0] * cal_m + 
             weights_norm[:, 1] * cal_l + 
             weights_norm[:, 2] * cal_i)
    return fused

def fuse_quality_threshold(cal_m, cal_l, cal_i, q_m, q_l, q_i, base_w_m, base_w_l, base_w_i, threshold=0.3):
    """Drop modality if quality < threshold, renormalize."""
    fused = np.zeros(len(cal_m))
    for idx in range(len(cal_m)):
        ws = [base_w_m if q_m[idx] >= threshold else 0,
              base_w_l if q_l[idx] >= threshold else 0,
              base_w_i if q_i[idx] >= threshold else 0]
        total = sum(ws)
        if total < 1e-10:
            # Fallback to equal weights
            ws = [1/3, 1/3, 1/3]
        else:
            ws = [w / total for w in ws]
        fused[idx] = ws[0] * cal_m[idx] + ws[1] * cal_l[idx] + ws[2] * cal_i[idx]
    return fused

def fuse_quality_linear(cal_m, cal_l, cal_i, q_m, q_l, q_i, base_w_m, base_w_l, base_w_i):
    """Linear scaling: w_m = base_w_m * (q_m / mean_q_m)."""
    mean_q_m = q_m.mean()
    mean_q_l = q_l.mean()
    mean_q_i = q_i.mean()
    
    w_m = base_w_m * (q_m / mean_q_m)
    w_l = base_w_l * (q_l / mean_q_l)
    w_i = base_w_i * (q_i / mean_q_i)
    
    # Normalize to sum to 1
    total = w_m + w_l + w_i
    w_m = w_m / total
    w_l = w_l / total
    w_i = w_i / total
    
    fused = w_m * cal_m + w_l * cal_l + w_i * cal_i
    return fused

# Evaluate all strategies
strategies = {
    'E027_fixed': lambda: fuse_fixed(cal_mo, cal_lo, cal_io, W_M_BASE, W_L_BASE, W_I_BASE),
    'quality_softmax': lambda: fuse_quality_softmax(cal_mo, cal_lo, cal_io, q_mfcc, q_audio, q_image, W_M_BASE, W_L_BASE, W_I_BASE),
    'quality_threshold_0.3': lambda: fuse_quality_threshold(cal_mo, cal_lo, cal_io, q_mfcc, q_audio, q_image, W_M_BASE, W_L_BASE, W_I_BASE, threshold=0.3),
    'quality_threshold_0.5': lambda: fuse_quality_threshold(cal_mo, cal_lo, cal_io, q_mfcc, q_audio, q_image, W_M_BASE, W_L_BASE, W_I_BASE, threshold=0.5),
    'quality_linear': lambda: fuse_quality_linear(cal_mo, cal_lo, cal_io, q_mfcc, q_audio, q_image, W_M_BASE, W_L_BASE, W_I_BASE),
}

results = []
for name, fuse_fn in strategies.items():
    fused = fuse_fn()
    eer, eer_thr = compute_eer(fused[y_all == 1], fused[y_all == 0])
    min_dcf, dc_thr = compute_min_dcf(fused[y_all == 1], fused[y_all == 0])
    results.append({'strategy': name, 'eer': eer * 100, 'min_dcf': min_dcf, 'threshold': dc_thr})
    print(f"{name:25s}: EER={eer*100:5.2f}%  min-DCF={min_dcf:.4f}")

results_df = pd.DataFrame(results)
print("\nSummary:")
print(results_df.sort_values('eer'))

## 6. Analysis: when does quality-aware fusion help?

In [ ]:
# Analyze samples where quality-aware fusion differs from fixed
fused_fixed = fuse_fixed(cal_mo, cal_lo, cal_io, W_M_BASE, W_L_BASE, W_I_BASE)
fused_q = fuse_quality_linear(cal_mo, cal_lo, cal_io, q_mfcc, q_audio, q_image, W_M_BASE, W_L_BASE, W_I_BASE)

# Find samples where quality-weighted fusion changed the decision
_, thr_fixed = compute_min_dcf(fused_fixed[y_all == 1], fused_fixed[y_all == 0])
_, thr_q = compute_min_dcf(fused_q[y_all == 1], fused_q[y_all == 0])

dec_fixed = (fused_fixed >= thr_fixed).astype(int)
dec_q = (fused_q >= thr_q).astype(int)

changed = dec_fixed != dec_q
print(f"Decisions changed: {changed.sum()} / {len(changed)} samples ({100*changed.mean():.1f}%)")

# Check if changed samples have lower quality
if changed.sum() > 0:
    print(f"\nChanged samples quality:")
    print(f"  Image quality: {q_image[changed].mean():.3f} vs {q_image[~changed].mean():.3f} (all)")
    print(f"  Audio quality: {q_audio[changed].mean():.3f} vs {q_audio[~changed].mean():.3f} (all)")

# Check error rates on changed samples
y_changed = y_all[changed]
pred_changed = dec_q[changed]
error_rate_changed = (pred_changed != y_changed).mean()
print(f"\nError rate on changed samples: {error_rate_changed*100:.1f}%")

## 7. Conclusion

Quality-aware fusion results:
- Best strategy: [TBD]
- EER improvement: [TBD] pp vs E027 fixed baseline
- Key insight: [TBD]

**Next steps:**
- If quality-aware fusion helps: integrate into predict_fusion.py
- If not: quality metrics may need refinement (e.g., better SNR estimator, face detection quality)